# Deals Data Cleaning

This notebook prepares the CRM **Deals** table, the central dataset used for funnel, sales, marketing, and product analysis.

### Main tasks
- remove CRM and technical duplicates
- standardize dates, monetary fields, and SLA response time
- flag date and payment inconsistencies
- handle missing values without erasing meaningful CRM states
- create analytical fields such as deal duration, buyer status, and funnel stage groups
- normalize German-language level and city values

> **Data note:** the original CRM files are not included in the public repository.


In [ ]:
from pathlib import Path
import datetime
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import to_snake, df_overview, df_clean_summary, colors

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.max_columns = None


## 1. Load and Initial Inspection

In [ ]:
deals = pd.read_excel(RAW_DIR / 'Deals (Done).xlsx', dtype={'Contact Name': str, 'Id': str})

In [ ]:
# Standardize column names to snake_case
deals.columns = [to_snake(c) for c in deals.columns]

In [ ]:
# Display a compact data-quality overview
df_overview(deals)

## 2. Remove Duplicate and Invalid Records

Three types of records are handled:
1. CRM records explicitly marked with `lost_reason = 'Duplicate'`
2. exact technical duplicate rows
3. empty records without a deal ID

In [ ]:
# Remove records explicitly marked as CRM duplicates
crm_duplicates = deals['lost_reason'].eq('Duplicate')
print(f'CRM duplicates (lost_reason=Duplicate): {crm_duplicates.sum()}')
deals = deals.loc[~crm_duplicates].copy()

# Remove exact technical duplicates
before_exact = len(deals)
deals = deals.drop_duplicates()
print(f'Exact technical duplicates removed: {before_exact - len(deals)}')

# Remove empty rows without a deal ID
before_missing_id = len(deals)
deals = deals.dropna(subset=['id'])
print(f'Rows without deal ID removed: {before_missing_id - len(deals)}')

In [ ]:
print(f'Rows after deduplication: {len(deals)}')

## 3. Data Types

### 3.1. `created_time`, `closing_date`

In [ ]:
# Parse creation and closing dates
deals['created_time'] = pd.to_datetime(
    deals['created_time'],
    format='%d.%m.%Y %H:%M',
    errors='coerce'
)
deals['closing_date'] = pd.to_datetime(
    deals['closing_date'],
    format='%d.%m.%Y',
    errors='coerce'
)

### 3.2. `sla`

In [ ]:
# Inspect raw SLA values
deals['sla'].head(20)

In [ ]:
# Inspect the underlying Python types stored in SLA
print(deals['sla'].apply(lambda x: type(x).__name__).value_counts())

The SLA column contains several underlying time representations. They are normalized to one metric: **response time in minutes**.

In [ ]:
def parse_sla(value):
    """Convert supported SLA time representations to minutes."""
    if isinstance(value, datetime.time):
        return round(
            (value.hour * 3600 + value.minute * 60 + value.second) / 60
        )

    if isinstance(value, (pd.Timedelta, datetime.timedelta)):
        return round(value.total_seconds() / 60)

    return np.nan


deals['sla_minutes'] = deals['sla'].apply(parse_sla).astype('Int64')

print(deals['sla_minutes'].describe())
print(f"Missing SLA values: {deals['sla_minutes'].isna().sum()}")

The minimum values reflect leads contacted within seconds. Extremely large SLA values are retained in the cleaned dataset and handled separately during analysis.

In [ ]:
deals = deals.drop(columns=['sla'])

### 3.3. Monetary Fields

In [ ]:
def clean_amount(value):
    """Parse CRM monetary values stored in European number format."""
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = value.replace('€', '').replace(' ', '')
    value = value.replace('.', '').replace(',', '.')

    return pd.to_numeric(value, errors='coerce')


deals['initial_amount_paid'] = deals['initial_amount_paid'].apply(clean_amount)
deals['offer_total_amount'] = deals['offer_total_amount'].apply(clean_amount)

In [ ]:
print('Data types after conversion:')
print(deals.dtypes)

### 3.4. `course_duration` and `months_of_study`

In [ ]:
deals['course_duration'].value_counts(dropna=False).sort_index()

In [ ]:
deals['months_of_study'].value_counts(dropna=False).sort_index()

In [ ]:
# Treat course length and months studied as discrete categorical values
deals['course_duration'] = deals['course_duration'].astype('Int64').astype('category')
deals['months_of_study'] = deals['months_of_study'].astype('Int64').astype('category')

print('course_duration type:', deals['course_duration'].dtype)
print('months_of_study type:', deals['months_of_study'].dtype)

## 4. Data Quality and Anomaly Flags

### 4.1. Closing Date Before Creation Date

A deal cannot logically close before it is created.

`closing_date` can contain manually entered CRM errors, while `created_time` is treated as the more reliable system timestamp. Instead of overwriting uncertain records, the notebook creates a status flag.

In [ ]:
# Classify date consistency before calculating deal duration
deals['duration_status'] = 'valid'

deals.loc[deals['created_time'].isna(), 'duration_status'] = 'missing_created_time'
deals.loc[deals['closing_date'].isna(), 'duration_status'] = 'not_closed'
deals.loc[
    deals['closing_date'].dt.date < deals['created_time'].dt.date,
    'duration_status'
] = 'anomaly'

same_day = deals['closing_date'].dt.date == deals['created_time'].dt.date
deals.loc[same_day, 'duration_status'] = 'same_day'

In [ ]:
deals['duration_status'].value_counts()

In [ ]:
# Calculate duration only when the date sequence is valid
valid_duration = deals['duration_status'].isin(['valid', 'same_day'])
deals['deal_duration_days'] = np.nan

deals.loc[valid_duration, 'deal_duration_days'] = (
    deals.loc[valid_duration, 'closing_date'].dt.normalize()
    - deals.loc[valid_duration, 'created_time'].dt.normalize()
).dt.days

In [ ]:
print('Deal duration range (days):')
print(
    f"{deals['deal_duration_days'].min()} - "
    f"{deals['deal_duration_days'].max()}"
)

### 4.2. `initial_amount_paid` > `offer_total_amount`

In [ ]:
payment_exceeds_offer = (
    deals['initial_amount_paid'] > deals['offer_total_amount']
)
print(
    'Rows with initial_amount_paid > offer_total_amount:',
    payment_exceeds_offer.sum()
)

A first payment should not normally exceed the total offer amount. These records are flagged for later buyer-level review.

In [ ]:
paid_without_study = (
    (deals['initial_amount_paid'] > 0)
    & deals['months_of_study'].isna()
)
result = deals.loc[paid_without_study]
print('Rows:', result.shape[0])

### 4.3. Positive Initial Payment but Missing Study Duration

A positive initial payment with no study duration may indicate payment before the course started or incomplete CRM data. The record is retained and flagged.

In [ ]:
payment_done_incomplete = (
    deals['stage'].eq('Payment Done')
    & (
        deals['initial_amount_paid'].isna()
        | deals['initial_amount_paid'].eq(0)
    )
    & (
        deals['months_of_study'].isna()
        | deals['months_of_study'].eq(0)
    )
)
result = deals.loc[payment_done_incomplete]
print('Rows:', result.shape[0])

### 4.4. `Payment Done` with Incomplete Payment / Study Data

According to the project documentation, `Payment Done` confirms that payment was received, but payment amount or study details can still be missing in the CRM.

Revenue-related analysis later uses confirmed buyer records with sufficient payment and study information.

In [ ]:
demo_amount = deals['initial_amount_paid'].isin([0, 1, 9])
result = deals.loc[demo_amount]
print('Rows with demo / symbolic payment amounts:', result.shape[0])

### 4.5. Small Payment Amounts (0, 1, 9)

In [ ]:
# Create a payment-quality flag while retaining the original records
conditions = [
    deals['initial_amount_paid'].isin([0, 1, 9]),
    deals['initial_amount_paid'] > deals['offer_total_amount'],
    (deals['initial_amount_paid'] > 0) & deals['months_of_study'].isna(),
    deals['stage'].eq('Payment Done')
    & (
        deals['initial_amount_paid'].isna()
        | deals['months_of_study'].isna()
    ),
]

choices = [
    'demo',
    'initial_more_than_offer',
    'paid_without_months',
    'payment_done_incomplete',
]

deals['payment_flag'] = np.select(
    conditions,
    choices,
    default='other'
)

print(deals['payment_flag'].value_counts())

Payment inconsistencies are retained in the main Deals dataset and captured through a dedicated flag. More detailed corrections are performed only where evidence is sufficiently strong in the buyer-level dataset.

In [ ]:
def outlier(column, k=1.5):
    """Return an IQR-based outlier mask for one Deals column."""
    q1 = deals[column].quantile(0.25)
    q3 = deals[column].quantile(0.75)
    iqr = q3 - q1

    return (
        (deals[column] < q1 - k * iqr)
        | (deals[column] > q3 + k * iqr)
    )

### 4.6. Outlier Review

In [ ]:
for column in numeric_cols:
    column_outliers = outlier(column)
    print(f"\n{'=' * 60}")
    print(f"Column: {column} | Outliers: {column_outliers.sum()}")
    print('=' * 60)
    display(deals.loc[column_outliers, ['id', column]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# SLA
sns.histplot(deals['sla_minutes'], bins=30, kde=True, color=colors['accent'], ax=axes[0])

axes[0].set_title('SLA Minutes Distribution')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Count')

# Deal Duration
sns.histplot(deals['deal_duration_days'], bins=30, kde=True, color=colors['accent'], ax=axes[1])

axes[1].set_title('Deal Duration Distribution')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
deals[['sla_minutes', 'deal_duration_days']].describe()

Both `sla_minutes` and `deal_duration_days` are right-skewed with long tails.

For downstream comparisons, the **99th percentile** is used as an analytical cutoff for extreme observations. The underlying records remain in the cleaned dataset.

In [ ]:
sla_99 = deals['sla_minutes'].quantile(0.99)
duration_99 = deals['deal_duration_days'].quantile(0.99)

print(
    f'SLA 99th percentile: {sla_99:.0f} min '
    f'({sla_99 / 60:.1f} hours)'
)
print(
    f'Deal duration 99th percentile: '
    f'{duration_99:.0f} days'
)

## 5. Missing Values

Missing values are handled according to business meaning rather than being filled mechanically.

**Retained as `NaN` when absence is meaningful**
- `lost_reason`: blank means the deal was not recorded as lost
- `source`, `campaign`: acquisition information may genuinely be unavailable
- `payment_type`: often populated only when payment details are entered
- `closing_date`: the potential deal may still be open
- `months_of_study`, `course_duration`: the lead may not have become a student
- `product`, `education_type`: may be unavailable at early funnel stages
- `content`, `term`: advertising metadata may not exist for non-ad traffic

**Eligible for `Unknown` when the attribute is expected but missing**
- `city`
- `level_of_deutsch`
- `quality`
- `page`

A contact may have more than one potential deal. Before filling contact attributes, the notebook checks whether values are consistent across deals for the same contact.

In [ ]:
fill_by_contact = ['source', 'campaign', 'city', 'level_of_deutsch', 'deal_owner_name']

In [ ]:
# Check how often the same contact has conflicting values
for col in fill_by_contact:
    inconsistent = deals.groupby('contact_name')[col].nunique()
    print(
        f'{col}: contacts with conflicting values: '
        f'{(inconsistent > 1).sum()}'
    )

`city` and `level_of_deutsch` show sufficiently low within-contact inconsistency for limited forward/backward filling.

`deal_owner_name` can legitimately change when a lead is transferred between managers. `source` and `campaign` also vary across repeated contacts and therefore are **not** backfilled, preserving acquisition history.

In [ ]:
# Backfill only attributes that are sufficiently stable within contact history
fill_cols = ['city', 'level_of_deutsch']
before = {col: deals[col].isna().sum() for col in fill_cols}

deals = deals.sort_values(['contact_name', 'created_time'])

for col in fill_cols:
    deals[col] = (
        deals.groupby('contact_name')[col]
        .transform(lambda x: x.ffill().bfill())
    )

for col in fill_cols:
    after = deals[col].isna().sum()
    print(
        f'{col}: missing before={before[col]}, '
        f'after={after}, filled={before[col] - after}'
    )

In [ ]:
# Use 'Unknown' only for attributes expected to exist but still missing
fill_unknown = ['city', 'level_of_deutsch', 'quality', 'page']

for col in fill_unknown:
    missing_count = deals[col].isna().sum()
    if missing_count:
        deals[col] = deals[col].fillna('Unknown')
        print(
            f'{col}: filled {missing_count} missing values with "Unknown"'
        )

## 6. Confirmed Buyer Flag

A confirmed buyer is defined conservatively as a record with **initial payment > 0** and **months of study > 0**.

In [ ]:
deals['is_buyer'] = (
    (deals['initial_amount_paid'] > 0)
    & (deals['months_of_study'].astype('Int64') > 0)
)

print(f"Confirmed buyer records: {deals['is_buyer'].sum()}")

In [ ]:
print(
    'Unique confirmed buyers:',
    deals.loc[deals['is_buyer'], 'contact_name'].nunique()
)

In [ ]:
# Compare Payment Done with the stricter analytical buyer definition

In [ ]:
payment_done = deals['stage'].eq('Payment Done').sum()
buyers = deals['is_buyer'].sum()

print(f'Payment Done: {payment_done}')
print(f'is_buyer: {buyers}')
print(f'Difference: {payment_done - buyers}')

In [ ]:
# Inspect Payment Done records excluded by the strict buyer definition

In [ ]:
deals[(deals['stage'] == 'Payment Done') & (~deals['is_buyer'])][['stage', 'initial_amount_paid', 'months_of_study', 'contact_name']]

`Payment Done` records and the stricter `is_buyer` definition do not fully overlap.

The project identified three main exceptions:
1. payment amount equal to 0 despite study information
2. positive payment but study duration equal to 0 / missing
3. payment confirmed by CRM status but payment or study details not recorded

Therefore, `is_buyer` is intentionally treated as a strict analytical flag supported by both payment and study data.

## 7. Stage

In [ ]:
deals['stage'].unique()

### Grouping Detailed CRM Stages

The CRM contains multiple detailed stages. For high-level funnel analysis they are grouped into:

- **In Progress** — the sales team is still working with the lead
- **Won** — payment is confirmed (`Payment Done`)
- **Lost** — the lead was marked as lost

In [ ]:
stage_map = {
    'New Lead': 'In Progress',
    'Registered on Webinar': 'In Progress',
    'Registered on Offline Day': 'In Progress',
    'Need To Call': 'In Progress',
    'Need to Call - Sales': 'In Progress',
    'Need a consultation': 'In Progress',
    'Call Delayed': 'In Progress',
    'Qualificated': 'In Progress',
    'Test Sent': 'In Progress',
    'Free Education': 'In Progress',
    'Waiting For Payment': 'In Progress',
    'Payment Done': 'Won',
    'Lost': 'Lost',
}
deals['stage_group'] = deals['stage'].map(stage_map)
print(deals['stage_group'].value_counts())

`Waiting For Payment` remains **In Progress**, not Won. According to the project documentation, payment has not yet reached the account at this stage; confirmed payment is represented by `Payment Done`.

## 8. Level of Deutsch

In [ ]:
deals['level_of_deutsch'].unique()

In [ ]:
def normalize_german_level(value):
    """Extract standardized CEFR-like A/B/C levels from CRM text."""
    text = str(value).upper()

    # Normalize common Cyrillic look-alike characters to Latin letters
    text = (
        text.replace('А', 'A')
        .replace('Б', 'B')
        .replace('В', 'B')
        .replace('С', 'C')
    )

    match = re.search(r'[ABC][012]', text)
    if match:
        return match.group(0)

    return 'Unknown'

In [ ]:
deals['level_of_deutsch'] = deals['level_of_deutsch'].apply(
    normalize_german_level
)
print(deals['level_of_deutsch'].value_counts())

## 9. City

In [2]:
## City Normalization

In [ ]:
deals["city"].str.contains(",", na=False).sum()

In [ ]:
city_replacements = {
    "Vor Ebersbach 1, 77761 Schiltach": "Schiltach",
    "Karl-Liebknecht str. 24, Hildburghausen, Thüringen": "Hildburghausen",
    "Poland , Gdansk , Al. Grunwaldzka 7, ap. 1a": "Gdansk",
    "-": "Unknown",
    "Gdańsk": "Gdansk",
    "Nuenchritz": "Nünchritz",
    "Villingen-Schwenningen": "Villingen‑Schwenningen"

}

deals["city"] = (deals["city"].replace(city_replacements).str.strip())

## 10. Final Quality Check

In [ ]:
df_clean_summary(deals)

## 11. Save Processed Data

In [ ]:
# Pickle preserves parsed data types for downstream notebooks
deals.to_pickle(PROCESSED_DIR / 'deals_clean.pkl')

print('Saved: deals_clean.pkl')
print(f'Shape: {deals.shape}')

## 12. Cleaning Summary

**Source table:** CRM potential deals with 23 original fields.

| Step | Action | Result |
|---|---|---|
| 1 | Column standardization | Converted names to `snake_case` |
| 2 | Deduplication | Removed CRM duplicates, exact technical duplicates, and rows without a deal ID |
| 3 | Datetime conversion | Converted `created_time` and `closing_date` to datetime |
| 4 | SLA normalization | Converted mixed time formats into `sla_minutes` |
| 5 | Monetary cleaning | Parsed `initial_amount_paid` and `offer_total_amount` into numeric values |
| 6 | Date-quality flags | Created `duration_status` and valid `deal_duration_days` |
| 7 | Payment-quality flags | Created `payment_flag` for demo / inconsistent / incomplete payment records |
| 8 | Missing-value handling | Preserved meaningful missingness and used limited contact-level filling only where justified |
| 9 | Language normalization | Standardized German language levels to values such as A2 / B1 |
| 10 | City normalization | Converted selected address-like and inconsistent entries to standardized city names |
| 11 | Buyer definition | Created a conservative `is_buyer` flag based on payment + study evidence |
| 12 | Funnel grouping | Created `stage_group`: `In Progress`, `Won`, `Lost` |

### Output
- `deals_clean.pkl` — cleaned central CRM deals dataset
